# Chapter 33: Visual Front End Pipeline

<a href="../lite/lab/index.html?path=ch33_visual_frontend.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

A camera produces millions of pixels per frame. But your SLAM system does not need
millions of points. It needs 200 reliable, well tracked feature points. The visual
front end's job is to find those 200 points, track them across frames, and ruthlessly
reject the ones that are wrong.

```{admonition} What you will build
:class: tip

- Detect corner features using a Harris like corner score
- Match features between frames using descriptors and the ratio test
- Track features across 5 consecutive frames using KLT style prediction
- Reject outlier matches using RANSAC for robust pose estimation

**Real world application:** The visual front end is the most fragile part of visual SLAM. 200 good features beat 2000 bad ones. After this chapter, you will know how to extract, match, track, and filter the features that SLAM depends on.
```

## 33.1 Feature Detection

Good features are **corners**: image locations where intensity changes sharply in
two independent directions. Edges change in only one direction and are ambiguous
along the edge. Flat regions have no gradient at all.

The **Harris corner detector** computes a score from the structure tensor (the
outer product of gradients averaged over a patch):

$$M = \sum_{\text{patch}} \begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix}$$

The Harris score is $R = \det(M) - k\,(\text{trace}(M))^2$. Corners have large positive $R$,
edges have $R \approx 0$, and flat regions have small $R$.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(42)
img_size   = 120         # synthetic image size
n_blobs    = 8           # number of Gaussian blobs (create gradients)
patch_r    = 3           # patch radius for structure tensor
harris_k   = 0.04       # Harris k parameter (try 0.02, 0.06)
# ─────────────────────────────────────────────────────────────────────────────

# Create a synthetic image with blobs (corners where blobs overlap)
yy, xx = np.mgrid[0:img_size, 0:img_size].astype(float)
img = np.zeros((img_size, img_size))
for _ in range(n_blobs):
    cx_b = np.random.uniform(15, img_size - 15)
    cy_b = np.random.uniform(15, img_size - 15)
    sx = np.random.uniform(5, 20)
    sy = np.random.uniform(5, 20)
    amp = np.random.uniform(0.3, 1.0)
    img += amp * np.exp(-((xx - cx_b)**2 / (2*sx**2) + (yy - cy_b)**2 / (2*sy**2)))

# Add a few sharp rectangles for clear corners
img[30:50, 60:90] += 0.8
img[70:100, 20:45] += 0.6
img[15:35, 15:30] += 0.5

# Compute gradients (Sobel approximation)
Ix = np.zeros_like(img)
Iy = np.zeros_like(img)
Ix[:, 1:-1] = (img[:, 2:] - img[:, :-2]) / 2
Iy[1:-1, :] = (img[2:, :] - img[:-2, :]) / 2

# Structure tensor components
Ixx = Ix * Ix
Iyy = Iy * Iy
Ixy = Ix * Iy

# Box filter (sum over patch)
from scipy.ndimage import uniform_filter
Sxx = uniform_filter(Ixx, size=2*patch_r+1)
Syy = uniform_filter(Iyy, size=2*patch_r+1)
Sxy = uniform_filter(Ixy, size=2*patch_r+1)

# Harris response
det_M = Sxx * Syy - Sxy**2
trace_M = Sxx + Syy
harris_resp = det_M - harris_k * trace_M**2

# Classify pixels
threshold_corner = harris_resp.max() * 0.01
threshold_edge   = harris_resp.max() * 0.001

# Non maximum suppression: find local maxima
from scipy.ndimage import maximum_filter
local_max = maximum_filter(harris_resp, size=7)
corners_mask = (harris_resp == local_max) & (harris_resp > threshold_corner)
corner_yx = np.argwhere(corners_mask)
corner_scores = harris_resp[corners_mask]

# Sort by score and take top 50
top_idx = np.argsort(corner_scores)[::-1][:50]
corner_yx = corner_yx[top_idx]
corner_scores = corner_scores[top_idx]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.imshow(img, cmap='gray', origin='upper')
ax.set_title('Synthetic image', fontsize=12)
ax.axis('off')

ax = axes[1]
ax.imshow(harris_resp, cmap='hot', origin='upper')
ax.set_title('Harris response (hot = corner)', fontsize=12)
ax.axis('off')

ax = axes[2]
ax.imshow(img, cmap='gray', origin='upper')
# Color features by quality score
sc = ax.scatter(corner_yx[:, 1], corner_yx[:, 0],
                c=corner_scores, cmap='RdYlGn', s=40,
                edgecolors='k', lw=0.5, zorder=5)
plt.colorbar(sc, ax=ax, label='Harris score', shrink=0.8)
ax.set_title(f'Detected features ({len(corner_yx)} corners)', fontsize=12)
ax.axis('off')

plt.tight_layout()
plt.show()

print(f'Detected {len(corner_yx)} corner features.')
print(f'Score range: [{corner_scores.min():.4f}, {corner_scores.max():.4f}]')

**Key observations:**
- Corners of the rectangles get the highest scores (gradients in two directions).
- Blob centres get low scores (gradients are weak at the peak).
- Non maximum suppression ensures features are well spread out.

## 33.2 Matching

Each detected feature is described by a **descriptor vector** that summarises the
local image patch. Matching finds pairs of features (one in each image) whose
descriptors are closest.

The **ratio test** (Lowe's test) rejects ambiguous matches: if the best match
distance is almost as small as the second best, the match is unreliable.

$$\text{reject if } \frac{d_1}{d_2} > \text{ratio\_threshold}$$

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(11)
n_feat        = 60       # features per image
desc_dim      = 32       # descriptor dimensionality
n_true_match  = 40       # true matches (inliers)
desc_noise    = 0.3      # noise on matched descriptors (try 0.1, 0.5)
ratio_thresh  = 0.75     # Lowe's ratio test threshold
# ─────────────────────────────────────────────────────────────────────────────

# Simulate descriptors
desc1 = np.random.randn(n_feat, desc_dim)
desc1 = desc1 / np.linalg.norm(desc1, axis=1, keepdims=True)  # unit vectors

# Image 2: first n_true_match are true matches (noisy copies)
desc2 = np.random.randn(n_feat, desc_dim)
desc2 = desc2 / np.linalg.norm(desc2, axis=1, keepdims=True)
# Replace first n_true_match with noisy versions of desc1
desc2[:n_true_match] = desc1[:n_true_match] + np.random.normal(0, desc_noise, (n_true_match, desc_dim))
desc2[:n_true_match] = desc2[:n_true_match] / np.linalg.norm(desc2[:n_true_match], axis=1, keepdims=True)

# Brute force matching: for each feature in image 1, find nearest in image 2
from scipy.spatial.distance import cdist
dists = cdist(desc1, desc2, metric='euclidean')

matches_raw = []
matches_ratio = []
for i in range(n_feat):
    sorted_j = np.argsort(dists[i])
    j_best   = sorted_j[0]
    d_best   = dists[i, j_best]
    d_second = dists[i, sorted_j[1]]
    matches_raw.append((i, j_best, d_best))
    if d_best / d_second < ratio_thresh:
        matches_ratio.append((i, j_best, d_best))

# Evaluate
def is_correct(i, j):
    return i < n_true_match and j == i  # true match means same index

raw_correct   = sum(1 for i, j, _ in matches_raw if is_correct(i, j))
ratio_correct = sum(1 for i, j, _ in matches_ratio if is_correct(i, j))

print(f'Raw matching: {len(matches_raw)} matches, {raw_correct} correct '
      f'({100*raw_correct/len(matches_raw):.0f}% precision)')
print(f'After ratio test: {len(matches_ratio)} matches, {ratio_correct} correct '
      f'({100*ratio_correct/max(len(matches_ratio),1):.0f}% precision)')

In [ ]:
# Visualise matches
# Simulate feature positions for visualisation
pos1 = np.random.uniform(20, 620, (n_feat, 2))
pos2 = pos1.copy()
pos2[:n_true_match] += np.random.normal(0, 10, (n_true_match, 2))  # true matches move slightly
pos2[n_true_match:] = np.random.uniform(20, 620, (n_feat - n_true_match, 2))  # wrong features

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax_idx, (match_list, title) in enumerate([
    (matches_raw, f'Raw matches ({len(matches_raw)})'),
    (matches_ratio, f'After ratio test ({len(matches_ratio)})')
]):
    ax = axes[ax_idx]
    ax.set_title(title, fontsize=12)
    for i, j, d in match_list:
        correct = is_correct(i, j)
        col = 'forestgreen' if correct else 'tomato'
        ax.plot([pos1[i, 0], pos2[j, 0] + 650],
                [pos1[i, 1], pos2[j, 1]],
                color=col, alpha=0.4, lw=0.8)
    ax.scatter(pos1[:, 0], pos1[:, 1], c='steelblue', s=15, zorder=5, label='image 1')
    ax.scatter(pos2[:, 0] + 650, pos2[:, 1], c='orange', s=15, zorder=5, label='image 2')
    ax.axvline(640, color='k', ls='--', alpha=0.3)
    ax.set_xlim(0, 1300); ax.set_ylim(0, 640)
    ax.legend(fontsize=9)
    ax.set_xlabel('x'); ax.set_ylabel('y')

plt.tight_layout()
plt.show()
print('Green = correct match, Red = wrong match')

**Key observations:**
- The ratio test dramatically improves **precision** (fraction of matches that are correct).
- It sacrifices some **recall** (some true matches are also rejected).
- The threshold 0.75 is a common default; lower values are more conservative.

## 33.3 Tracking

Instead of re detecting and matching features from scratch every frame, we can
**track** them using motion prediction. For each feature in frame $t$, we predict
where it should appear in frame $t+1$ using the current motion model, then search
a small window around the prediction.

This is the idea behind **KLT (Kanade Lucas Tomasi) tracking**.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(33)
n_track      = 40        # features to track
n_frames     = 5         # number of frames
true_dx      = 15.0      # true inter frame x displacement (px)
true_dy      = 5.0       # true inter frame y displacement (px)
motion_noise = 3.0       # noise on actual feature motion
search_radius = 10.0     # search window radius
lost_prob    = 0.05      # probability of losing a track per frame
# ─────────────────────────────────────────────────────────────────────────────

# Initial feature positions
positions = [np.random.uniform(50, 550, (n_track, 2))]
tracked_ids = np.arange(n_track)  # active track IDs
all_tracks = {i: [positions[0][i].copy()] for i in range(n_track)}

for frame in range(1, n_frames):
    prev = positions[-1].copy()
    # True motion + noise
    actual_motion = np.array([true_dx, true_dy]) + np.random.normal(0, motion_noise, (len(prev), 2))
    true_next = prev + actual_motion

    # Predicted position (using motion model estimate)
    predicted = prev + np.array([true_dx, true_dy])  # assume we know approximate motion

    # "Search" near prediction: find the true position if it is within search_radius
    found = np.linalg.norm(true_next - predicted, axis=1) < search_radius

    # Randomly lose some tracks
    lost = np.random.random(len(prev)) < lost_prob
    found = found & ~lost

    # Update positions
    new_pos = true_next[found]
    new_ids = tracked_ids[found]

    for idx, tid in enumerate(new_ids):
        all_tracks[tid].append(new_pos[idx].copy())

    positions.append(new_pos)
    tracked_ids = new_ids

print(f'Started with {n_track} features.')
print(f'After {n_frames-1} frames: {len(tracked_ids)} features still tracked.')
print(f'Lost {n_track - len(tracked_ids)} features along the way.')

In [ ]:
# Visualise tracked features across all frames
fig, ax = plt.subplots(figsize=(12, 7))

# Color by track length
cmap = plt.cm.viridis
max_len = max(len(t) for t in all_tracks.values())

for tid, track in all_tracks.items():
    track = np.array(track)
    n_seg = len(track)
    color = cmap(n_seg / max_len)
    ax.plot(track[:, 0], track[:, 1], '-', color=color, lw=1, alpha=0.7)
    ax.scatter(track[0, 0], track[0, 1], c='steelblue', s=20, zorder=5)
    if n_seg > 1:
        ax.scatter(track[-1, 0], track[-1, 1], c='tomato', s=20, zorder=5, marker='x')

# Legend
ax.scatter([], [], c='steelblue', s=30, label='start position')
ax.scatter([], [], c='tomato', s=30, marker='x', label='end position')
ax.plot([], [], color=cmap(1.0), lw=2, label=f'full track ({n_frames} frames)')
ax.plot([], [], color=cmap(0.3), lw=2, label='short track (lost early)')

ax.set_xlabel('x (pixels)'); ax.set_ylabel('y (pixels)')
ax.set_title(f'Feature tracking over {n_frames} frames ({len(tracked_ids)}/{n_track} survived)',
             fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Tracking statistics

How many features survive each frame? A healthy front end maintains at least 100 tracks
at all times. If the count drops, new features must be detected.

In [ ]:
frame_counts = [n_track] + [len(p) for p in positions[1:]]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(n_frames), frame_counts, color='steelblue', edgecolor='k', lw=0.5)
ax.set_xlabel('Frame'); ax.set_ylabel('Active tracks')
ax.set_title('Number of tracked features per frame', fontsize=12)
ax.set_xticks(range(n_frames))
for i, c in enumerate(frame_counts):
    ax.text(i, c + 0.5, str(c), ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 33.4 Outlier Rejection with RANSAC

Even after the ratio test and tracking, some matches are wrong. **RANSAC**
(Random Sample Consensus) robustly estimates a geometric model despite outliers.

Algorithm:
1. **Sample** a minimal set of correspondences (2 points for a 2D rigid transform).
2. **Estimate** the model (rotation + translation) from the minimal set.
3. **Count** how many other correspondences agree (inliers).
4. **Repeat** many times and keep the model with the most inliers.
5. **Refit** the model using all inliers.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(42)
n_inliers      = 40
n_outliers     = 15
true_angle_deg = 5.0          # true rotation (degrees)
true_tx, true_ty = 12.0, 8.0  # true translation (pixels)
inlier_noise   = 1.5          # noise on inlier correspondences
ransac_iters   = 200          # RANSAC iterations
inlier_thresh  = 5.0          # inlier distance threshold (pixels)
# ─────────────────────────────────────────────────────────────────────────────

# True rigid transform
theta_true = np.radians(true_angle_deg)
R_true = np.array([[np.cos(theta_true), -np.sin(theta_true)],
                   [np.sin(theta_true),  np.cos(theta_true)]])
t_true = np.array([true_tx, true_ty])

# Source points
n_total = n_inliers + n_outliers
src = np.random.uniform(50, 550, (n_total, 2))

# Destination: inliers follow the true transform + noise
dst = np.zeros_like(src)
dst[:n_inliers] = (R_true @ src[:n_inliers].T).T + t_true + \
                  np.random.normal(0, inlier_noise, (n_inliers, 2))
# Outliers: random positions
dst[n_inliers:] = np.random.uniform(50, 550, (n_outliers, 2))

def estimate_rigid_2d(p1, p2):
    """Estimate 2D rigid transform from 2 point pairs."""
    # Centres
    c1 = p1.mean(axis=0); c2 = p2.mean(axis=0)
    q1 = p1 - c1; q2 = p2 - c2
    # Rotation
    H = q1.T @ q2
    U, _, Vt = np.linalg.svd(H)
    R_est = Vt.T @ U.T
    if np.linalg.det(R_est) < 0:
        Vt[-1] *= -1
        R_est = Vt.T @ U.T
    t_est = c2 - R_est @ c1
    return R_est, t_est

# RANSAC
best_inlier_mask = np.zeros(n_total, dtype=bool)
best_R, best_t = np.eye(2), np.zeros(2)
inlier_history = []

for iteration in range(ransac_iters):
    # Sample 2 random correspondences
    idx = np.random.choice(n_total, 2, replace=False)
    R_hyp, t_hyp = estimate_rigid_2d(src[idx], dst[idx])

    # Compute residuals for all points
    dst_pred = (R_hyp @ src.T).T + t_hyp
    residuals = np.linalg.norm(dst - dst_pred, axis=1)
    inlier_mask = residuals < inlier_thresh

    n_in = inlier_mask.sum()
    inlier_history.append(n_in)

    if n_in > best_inlier_mask.sum():
        best_inlier_mask = inlier_mask.copy()
        best_R, best_t = R_hyp, t_hyp

# Refit using all inliers
if best_inlier_mask.sum() >= 2:
    best_R, best_t = estimate_rigid_2d(src[best_inlier_mask], dst[best_inlier_mask])

est_angle = np.degrees(np.arctan2(best_R[1, 0], best_R[0, 0]))

print(f'RANSAC results after {ransac_iters} iterations:')
print(f'  Inliers found: {best_inlier_mask.sum()} / {n_total}')
print(f'  True angle: {true_angle_deg:.1f} deg, Estimated: {est_angle:.2f} deg')
print(f'  True t: [{true_tx}, {true_ty}], Estimated: [{best_t[0]:.2f}, {best_t[1]:.2f}]')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Before RANSAC
ax = axes[0]
ax.set_title('All correspondences', fontsize=12)
for i in range(n_total):
    ax.plot([src[i,0], dst[i,0]], [src[i,1], dst[i,1]], color='gray', alpha=0.3)
ax.scatter(src[:, 0], src[:, 1], c='steelblue', s=20, zorder=5, label='source')
ax.scatter(dst[:, 0], dst[:, 1], c='orange', s=20, zorder=5, label='destination')
ax.legend(fontsize=9)

# After RANSAC
ax = axes[1]
ax.set_title(f'After RANSAC: {best_inlier_mask.sum()} inliers, '
             f'{n_total - best_inlier_mask.sum()} outliers', fontsize=11)
for i in range(n_total):
    col = 'forestgreen' if best_inlier_mask[i] else 'tomato'
    ax.plot([src[i,0], dst[i,0]], [src[i,1], dst[i,1]], color=col, alpha=0.5, lw=0.8)
ax.scatter(src[best_inlier_mask, 0], src[best_inlier_mask, 1],
           c='forestgreen', s=25, zorder=5, label='inlier')
ax.scatter(src[~best_inlier_mask, 0], src[~best_inlier_mask, 1],
           c='tomato', s=25, zorder=5, marker='x', label='outlier')
ax.legend(fontsize=9)

# Iteration history
ax = axes[2]
ax.plot(inlier_history, color='steelblue', lw=0.5, alpha=0.7)
best_so_far = np.maximum.accumulate(inlier_history)
ax.plot(best_so_far, color='tomato', lw=2, label='best so far')
ax.axhline(n_inliers, color='forestgreen', ls='--', label=f'true inliers ({n_inliers})')
ax.set_xlabel('iteration'); ax.set_ylabel('inlier count')
ax.set_title('RANSAC convergence', fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

**Key observations:**
- RANSAC correctly separates inliers from outliers even with 25% contamination.
- The estimated transform closely matches the true one.
- More iterations improve the probability of finding the best model but with diminishing returns.

### RANSAC theory: how many iterations do we need?

If the inlier ratio is $w$ and we sample $s$ points per iteration, the probability
that at least one iteration draws all inliers after $k$ iterations is:

$$P = 1 - (1 - w^s)^k$$

Solving for $k$:

$$k = \frac{\log(1 - P)}{\log(1 - w^s)}$$

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
s = 2            # minimal sample size
p_success = 0.99 # desired success probability
# ─────────────────────────────────────────────────────────────────────────────

w_values = np.linspace(0.3, 0.95, 50)
k_needed = np.log(1 - p_success) / np.log(1 - w_values**s)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(w_values * 100, k_needed, 'o-', color='steelblue', ms=3)
ax.set_xlabel('Inlier ratio (%)')
ax.set_ylabel('Required iterations')
ax.set_title(f'RANSAC iterations needed (s={s}, P={p_success})', fontsize=13)
ax.set_yscale('log')
ax.axhline(100, color='tomato', ls='--', alpha=0.5, label='100 iterations')
ax.legend()
plt.tight_layout()
plt.show()

# Print a few values
for w in [0.3, 0.5, 0.7, 0.9]:
    k = np.log(1 - p_success) / np.log(1 - w**s)
    print(f'  Inlier ratio {w*100:.0f}%: need {int(np.ceil(k))} iterations')

## Capstone: Complete Visual Front End Pipeline

We now simulate the full pipeline over 5 consecutive frames:
1. **Detect** features in the first frame.
2. For each subsequent frame, **predict** feature positions using the motion model.
3. **Match** predictions to actual positions (with noise and outliers).
4. Run **RANSAC** to reject outliers and estimate the inter frame rigid transform.
5. Report statistics at each step.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(2024)
n_initial     = 80       # initial features
n_cap_frames  = 5        # number of frames
motion_per_frame = np.array([20.0, 5.0])  # true inter frame translation
rot_per_frame    = 3.0   # degrees per frame
track_noise      = 2.0   # pixel noise on tracked features
outlier_frac     = 0.15  # fraction of outliers injected per frame
ransac_k         = 150   # RANSAC iterations
ransac_thresh    = 6.0   # inlier threshold
# ─────────────────────────────────────────────────────────────────────────────

# Frame 0: detect features
features = [np.random.uniform(30, 600, (n_initial, 2))]
labels   = [np.arange(n_initial)]  # track IDs
next_id  = n_initial

step_reports = []
all_frame_tracks = {i: [features[0][i].copy()] for i in range(n_initial)}

cumulative_R = np.eye(2)
cumulative_t = np.zeros(2)
estimated_poses = [(np.eye(2).copy(), np.zeros(2).copy())]

for frame in range(1, n_cap_frames):
    prev_pts = features[-1]
    prev_ids = labels[-1]
    n_prev = len(prev_pts)

    # True motion for this frame
    angle = np.radians(rot_per_frame)
    R_frame = np.array([[np.cos(angle), -np.sin(angle)],
                        [np.sin(angle),  np.cos(angle)]])
    t_frame = motion_per_frame.copy()

    # True next positions (inliers)
    true_next = (R_frame @ prev_pts.T).T + t_frame
    tracked = true_next + np.random.normal(0, track_noise, true_next.shape)

    # Inject outliers
    n_out = int(n_prev * outlier_frac)
    outlier_idx = np.random.choice(n_prev, n_out, replace=False)
    tracked[outlier_idx] = np.random.uniform(30, 600, (n_out, 2))

    # RANSAC
    best_mask = np.zeros(n_prev, dtype=bool)
    best_R_est = np.eye(2)
    best_t_est = np.zeros(2)

    for _ in range(ransac_k):
        idx = np.random.choice(n_prev, 2, replace=False)
        try:
            Rh, th = estimate_rigid_2d(prev_pts[idx], tracked[idx])
        except:
            continue
        pred = (Rh @ prev_pts.T).T + th
        res = np.linalg.norm(tracked - pred, axis=1)
        mask = res < ransac_thresh
        if mask.sum() > best_mask.sum():
            best_mask = mask.copy()
            best_R_est, best_t_est = Rh, th

    # Refit
    if best_mask.sum() >= 2:
        best_R_est, best_t_est = estimate_rigid_2d(prev_pts[best_mask], tracked[best_mask])

    # Keep only inliers
    inlier_pts = tracked[best_mask]
    inlier_ids = prev_ids[best_mask]

    # Update cumulative pose
    cumulative_R = best_R_est @ cumulative_R
    cumulative_t = best_R_est @ cumulative_t + best_t_est
    estimated_poses.append((cumulative_R.copy(), cumulative_t.copy()))

    # Update tracks
    for idx_in, tid in enumerate(inlier_ids):
        if tid in all_frame_tracks:
            all_frame_tracks[tid].append(inlier_pts[idx_in].copy())

    features.append(inlier_pts)
    labels.append(inlier_ids)

    est_ang = np.degrees(np.arctan2(best_R_est[1,0], best_R_est[0,0]))
    step_reports.append({
        'frame': frame,
        'detected': n_prev,
        'matched': n_prev,
        'inliers': best_mask.sum(),
        'outliers': n_prev - best_mask.sum(),
        'est_angle': est_ang,
        'est_tx': best_t_est[0],
        'est_ty': best_t_est[1]
    })

# Print report
print(f'{"Frame":>6} {"Detected":>9} {"Matched":>8} {"Inliers":>8} {"Outliers":>9} '
      f'{"Angle":>8} {"tx":>8} {"ty":>8}')
print('-' * 75)
for r in step_reports:
    print(f"{r['frame']:>6d} {r['detected']:>9d} {r['matched']:>8d} {r['inliers']:>8d} "
          f"{r['outliers']:>9d} {r['est_angle']:>8.2f} {r['est_tx']:>8.2f} {r['est_ty']:>8.2f}")
print(f'\nTrue motion per frame: angle={rot_per_frame} deg, '
      f'tx={motion_per_frame[0]}, ty={motion_per_frame[1]}')

In [ ]:
# Visualise tracked features across all frames with connecting lines
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: feature tracks
ax = axes[0]
frame_colors = ['steelblue', 'tomato', 'orange', 'forestgreen', 'purple']

for tid, track in all_frame_tracks.items():
    if len(track) < 2:
        continue
    track = np.array(track)
    ax.plot(track[:, 0], track[:, 1], '-', color='gray', alpha=0.3, lw=0.8)

# Plot features per frame
for f_idx in range(n_cap_frames):
    pts = features[f_idx]
    ax.scatter(pts[:, 0], pts[:, 1], c=frame_colors[f_idx % len(frame_colors)],
              s=15, alpha=0.7, label=f'frame {f_idx} ({len(pts)} pts)')

ax.set_xlabel('x (pixels)'); ax.set_ylabel('y (pixels)')
ax.set_title('Feature tracks across 5 frames', fontsize=13)
ax.legend(fontsize=8, loc='upper left')

# Right: pipeline statistics
ax = axes[1]
frames = [r['frame'] for r in step_reports]
inliers_list = [r['inliers'] for r in step_reports]
outliers_list = [r['outliers'] for r in step_reports]

width = 0.35
x_pos = np.arange(len(frames))
ax.bar(x_pos - width/2, inliers_list, width, color='forestgreen', label='inliers', edgecolor='k', lw=0.3)
ax.bar(x_pos + width/2, outliers_list, width, color='tomato', label='outliers', edgecolor='k', lw=0.3)
ax.set_xlabel('Frame transition')
ax.set_ylabel('Count')
ax.set_title('Inliers vs outliers per frame', fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{f-1} to {f}' for f in frames])
ax.legend()

plt.tight_layout()
plt.show()

**Capstone takeaways:**
- The pipeline successfully tracks features through multiple frames while rejecting outliers.
- RANSAC recovers the true inter frame motion accurately despite 15% outlier contamination.
- Feature count drops over time because some tracks are lost; in a real system, new features would be detected to replenish the pool.

---

## Exercises

### Exercise 33.1: Harris parameter sensitivity

Vary the Harris $k$ parameter from 0.01 to 0.1 in 10 steps. For each, run the
corner detector on the synthetic image from Section 33.1 and count the number of
detected corners (using the same threshold). Plot the count vs $k$. What value of
$k$ gives the most corners?

In [ ]:
# Your code here

### Exercise 33.2: Ratio test threshold sweep

Using the matching simulation from Section 33.2, sweep the ratio test threshold
from 0.5 to 0.95. Plot precision and recall vs threshold on the same axes.
At what threshold is precision = recall?

In [ ]:
# Your code here

### Exercise 33.3: RANSAC with varying outlier ratios

Run the RANSAC experiment from Section 33.4 with outlier fractions of 10%, 20%,
30%, 40%, and 50%. For each, record the rotation and translation error.
At what outlier fraction does RANSAC start failing?

In [ ]:
# Your code here

### Exercise 33.4: Feature replenishment

Modify the capstone pipeline so that whenever the feature count drops below 50,
new features are detected (simulated as random positions) to bring the total
back to 80. Run for 10 frames and plot the feature count over time.

In [ ]:
# Your code here

### Exercise 33.5: RANSAC for similarity transform (challenge)

Extend the RANSAC implementation to estimate a **similarity transform**
(rotation + translation + uniform scale). The minimal sample is still 2 points.
Test with a true scale factor of 1.1 and 20% outliers. Report the estimated scale,
rotation, and translation.

In [ ]:
# Your code here